# Flight Delays Data Warehouse — OLAP Analysis

This notebook connects to the PostgreSQL data warehouse and executes six analytical SQL queries (Phase 6, Variant B).  
Each query is visualised with matplotlib / seaborn.

**Prerequisites:**  
1. `docker compose up -d` — PostgreSQL must be running  
2. ETL must have been executed (`python etl/etl.py`) at least once  
3. `pip install -r requirements.txt`


In [ ]:
import os
import warnings
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.figsize'] = (14, 6)

DB_CONFIG = dict(
    host=os.getenv('DB_HOST', 'localhost'),
    port=int(os.getenv('DB_PORT', 5432)),
    dbname=os.getenv('DB_NAME', 'flight_dw'),
    user=os.getenv('DB_USER', 'postgres'),
    password=os.getenv('DB_PASSWORD', 'postgres'),
)

conn = psycopg2.connect(**DB_CONFIG)
print('Connected to PostgreSQL:', DB_CONFIG['dbname'])

---
## Analysis 1 — Long-term Delay Trend by Airline

**Business question:** How has average departure delay evolved per airline across years and months?  
COVID-19 (2020–2021) should be visible as a drop in flight volume and/or delay patterns.

In [ ]:
SQL_1 = """
SELECT
    d.year,
    d.month,
    a.airline_name,
    COUNT(*)                                        AS total_flights,
    ROUND(AVG(f.departure_delay_min)::NUMERIC, 2)  AS avg_dep_delay_min
FROM Fact_Flight_Operations f
JOIN Dim_Date    d ON f.date_key    = d.date_key
JOIN Dim_Airline a ON f.airline_key = a.airline_key
WHERE f.cancelled_flag = FALSE
GROUP BY d.year, d.month, a.airline_name
ORDER BY a.airline_name, d.year, d.month
"""
df1 = pd.read_sql_query(SQL_1, conn)
df1['year_month'] = df1['year'].astype(str) + '-' + df1['month'].astype(str).str.zfill(2)
df1.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Left: average delay
pivot_delay = df1.pivot_table(index='year_month', columns='airline_name',
                               values='avg_dep_delay_min', aggfunc='mean')
pivot_delay.plot(ax=axes[0], linewidth=1.2, alpha=0.8, legend=False)
axes[0].set_title('Average Departure Delay by Airline (per Month)')
axes[0].set_xlabel('Year-Month')
axes[0].set_ylabel('Avg Departure Delay (min)')
tick_count = max(1, len(pivot_delay) // 12)
axes[0].set_xticks(range(0, len(pivot_delay), tick_count))
axes[0].set_xticklabels(pivot_delay.index[::tick_count], rotation=45, ha='right')

# Right: total flights
pivot_flights = df1.pivot_table(index='year_month', columns='airline_name',
                                 values='total_flights', aggfunc='sum')
pivot_flights.plot(ax=axes[1], linewidth=1.2, alpha=0.8)
axes[1].set_title('Total Flights by Airline (per Month)')
axes[1].set_xlabel('Year-Month')
axes[1].set_ylabel('Number of Flights')
axes[1].set_xticks(range(0, len(pivot_flights), tick_count))
axes[1].set_xticklabels(pivot_flights.index[::tick_count], rotation=45, ha='right')
axes[1].legend(loc='upper right', fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig('notebooks/analysis_1_delay_trend.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Analysis 2 — Top 10 Most Delayed Routes

**Business question:** Which origin–destination pairs have the highest average departure delay (minimum 100 flights)?

In [ ]:
SQL_2 = """
SELECT
    orig.iata_code || ' → ' || dest.iata_code          AS route,
    orig.city || ' → ' || dest.city                    AS route_cities,
    COUNT(*)                                            AS total_flights,
    ROUND(AVG(f.departure_delay_min)::NUMERIC, 2)      AS avg_dep_delay_min
FROM Fact_Flight_Operations f
JOIN Dim_Airport orig ON f.origin_airport_key      = orig.airport_key
JOIN Dim_Airport dest ON f.destination_airport_key = dest.airport_key
WHERE f.cancelled_flag = FALSE AND f.departure_delay_min > 0
GROUP BY orig.iata_code, dest.iata_code, orig.city, dest.city
HAVING COUNT(*) >= 100
ORDER BY avg_dep_delay_min DESC
LIMIT 10
"""
df2 = pd.read_sql_query(SQL_2, conn)
df2

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(df2['route_cities'], df2['avg_dep_delay_min'],
               color=sns.color_palette('tab10', len(df2)))
ax.set_xlabel('Average Departure Delay (min)')
ax.set_title('Top 10 Most Delayed Routes (min. 100 flights, delayed only)')
ax.invert_yaxis()
for bar, val in zip(bars, df2['avg_dep_delay_min']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('notebooks/analysis_2_delayed_routes.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Analysis 3 — Cancellation Breakdown by Reason and Airline

**Business question:** Which airline cancels the most flights, and what is the dominant cancellation reason?

In [ ]:
SQL_3 = """
SELECT
    a.airline_name,
    r.description  AS cancellation_reason,
    COUNT(*)       AS cancelled_flights
FROM Fact_Flight_Operations f
JOIN Dim_Airline              a ON f.airline_key       = a.airline_key
JOIN Dim_Cancellation_Reason  r ON f.cancel_reason_key = r.cancel_reason_key
WHERE f.cancelled_flag = TRUE AND r.cancellation_code != 'N'
GROUP BY a.airline_name, r.description
ORDER BY a.airline_name, cancelled_flights DESC
"""
df3 = pd.read_sql_query(SQL_3, conn)
df3_pivot = df3.pivot_table(index='airline_name', columns='cancellation_reason',
                              values='cancelled_flights', fill_value=0)
df3_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
df3_pivot.plot(kind='bar', stacked=True, ax=ax, colormap='Set2')
ax.set_title('Cancellations by Airline and Reason')
ax.set_xlabel('Airline')
ax.set_ylabel('Number of Cancelled Flights')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Cancellation Reason', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('notebooks/analysis_3_cancellations.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Analysis 4 — Delay Cause Share per Airline

**Business question:** For delayed flights, what percentage of total delay minutes is attributable to each cause?

In [ ]:
SQL_4 = """
SELECT
    a.airline_name,
    SUM(COALESCE(f.carrier_delay_min,       0)) AS carrier,
    SUM(COALESCE(f.weather_delay_min,       0)) AS weather,
    SUM(COALESCE(f.nas_delay_min,           0)) AS nas,
    SUM(COALESCE(f.security_delay_min,      0)) AS security,
    SUM(COALESCE(f.late_aircraft_delay_min, 0)) AS late_aircraft
FROM Fact_Flight_Operations f
JOIN Dim_Airline a ON f.airline_key = a.airline_key
WHERE (
    COALESCE(f.carrier_delay_min, 0)
  + COALESCE(f.weather_delay_min, 0)
  + COALESCE(f.nas_delay_min, 0)
  + COALESCE(f.security_delay_min, 0)
  + COALESCE(f.late_aircraft_delay_min, 0)
) > 0
GROUP BY a.airline_name
ORDER BY (carrier + weather + nas + security + late_aircraft) DESC
"""
df4 = pd.read_sql_query(SQL_4, conn)
df4 = df4.set_index('airline_name')
df4_pct = df4.div(df4.sum(axis=1), axis=0) * 100
df4_pct

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
df4_pct.plot(kind='bar', stacked=True, ax=ax,
             color=['#4e79a7', '#76b7b2', '#f28e2b', '#e15759', '#59a14f'])
ax.set_title('Delay Cause Share per Airline (% of total delay minutes)')
ax.set_xlabel('Airline')
ax.set_ylabel('Share (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title='Delay Cause', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('notebooks/analysis_4_delay_causes.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Analysis 5 — Weekend vs Weekday Delays by Quarter

**Business question:** Are flights significantly more delayed (>15 min) on weekends vs weekdays, and does this vary by quarter?

In [ ]:
SQL_5 = """
SELECT
    d.quarter,
    CASE WHEN d.is_weekend THEN 'Weekend' ELSE 'Weekday' END  AS day_type,
    COUNT(*)                                                   AS total_flights,
    SUM(CASE WHEN f.departure_delay_min > 15 THEN 1 ELSE 0 END) AS significantly_delayed,
    ROUND(
        100.0
        * SUM(CASE WHEN f.departure_delay_min > 15 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0)::NUMERIC, 2
    )                                                          AS delayed_pct,
    ROUND(AVG(f.departure_delay_min)::NUMERIC, 2)             AS avg_dep_delay_min
FROM Fact_Flight_Operations f
JOIN Dim_Date d ON f.date_key = d.date_key
WHERE f.cancelled_flag = FALSE
GROUP BY d.quarter, d.is_weekend
ORDER BY d.quarter, d.is_weekend
"""
df5 = pd.read_sql_query(SQL_5, conn)
df5

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot5_pct = df5.pivot_table(index='quarter', columns='day_type', values='delayed_pct')
pivot5_avg = df5.pivot_table(index='quarter', columns='day_type', values='avg_dep_delay_min')

pivot5_pct.plot(kind='bar', ax=axes[0], color=['#4e79a7', '#f28e2b'])
axes[0].set_title('% Significantly Delayed Flights (>15 min) by Quarter')
axes[0].set_xlabel('Quarter')
axes[0].set_ylabel('% Delayed Flights')
axes[0].set_xticklabels([f'Q{q}' for q in range(1, 5)], rotation=0)
axes[0].legend(title='Day Type')

pivot5_avg.plot(kind='bar', ax=axes[1], color=['#4e79a7', '#f28e2b'])
axes[1].set_title('Average Departure Delay by Quarter')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Avg Departure Delay (min)')
axes[1].set_xticklabels([f'Q{q}' for q in range(1, 5)], rotation=0)
axes[1].legend(title='Day Type')

plt.tight_layout()
plt.savefig('notebooks/analysis_5_weekend_weekday.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Analysis 6 — Top 20 Airports: Volume, Delay, Cancellation Rate

**Business question:** How do the busiest airports compare in terms of average delay and cancellation rate?

In [ ]:
SQL_6 = """
SELECT
    ap.iata_code,
    ap.city,
    COUNT(*)                                                AS outbound_flights,
    ROUND(AVG(f.departure_delay_min)::NUMERIC, 2)          AS avg_dep_delay_min,
    SUM(CASE WHEN f.cancelled_flag THEN 1 ELSE 0 END)      AS total_cancellations,
    ROUND(
        100.0
        * SUM(CASE WHEN f.cancelled_flag THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0)::NUMERIC, 2
    )                                                       AS cancellation_rate_pct
FROM Fact_Flight_Operations f
JOIN Dim_Airport ap ON f.origin_airport_key = ap.airport_key
GROUP BY ap.iata_code, ap.city
ORDER BY outbound_flights DESC
LIMIT 20
"""
df6 = pd.read_sql_query(SQL_6, conn)
df6

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(
    df6['outbound_flights'],
    df6['avg_dep_delay_min'],
    s=df6['total_cancellations'] / 20,
    c=df6['cancellation_rate_pct'],
    cmap='YlOrRd',
    alpha=0.8,
    edgecolors='grey',
    linewidths=0.5,
)

for _, row in df6.iterrows():
    ax.annotate(
        row['iata_code'],
        (row['outbound_flights'], row['avg_dep_delay_min']),
        textcoords='offset points',
        xytext=(6, 3),
        fontsize=8,
    )

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cancellation Rate (%)')
ax.set_title('Top 20 Airports: Outbound Volume vs Average Delay\n(bubble size = total cancellations)')
ax.set_xlabel('Outbound Flights')
ax.set_ylabel('Average Departure Delay (min)')
plt.tight_layout()
plt.savefig('notebooks/analysis_6_airports.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
conn.close()
print('Connection closed.')